# 13. Modelo 3 – Regresión Lineal con Tendencia Temporal

El modelo de regresión lineal asume una relación lineal entre el año y la tasa de mortalidad.

- **Fortaleza:** Transparente, interpretable, bajo riesgo de sobreajuste
- **Limitación:** No captura aceleraciones o cambios de dirección. Para series no estacionarias (I(1)), sus intervalos de predicción son inválidos.

## 13.1. Diagnóstico de residuos – Regresión Lineal

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("NCHS_Leading_Causes.csv", dtype=str)
df.columns = ['year','cause_113','cause_name','state','deaths','age_adjusted_death_rate']
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['deaths'] = pd.to_numeric(df['deaths'].str.replace('.','',regex=False).str.replace(',','',regex=False), errors='coerce').fillna(0).astype(int)
df['age_adjusted_death_rate'] = pd.to_numeric(df['age_adjusted_death_rate'].str.replace(',','.'), errors='coerce')
df['state'] = df['state'].str.strip()
df['cause_name'] = df['cause_name'].str.strip()
df = df[(df['year']>=1999)&(df['year']<=2017)].dropna(subset=['year','age_adjusted_death_rate'])

estados = df[df['state']!='United States']
us = df[df['state']=='United States']
estados_2017 = estados[estados['year']==2017]
sin_all = estados[estados['cause_name']!='All causes']
print(f"Datos cargados: {len(df):,} filas | {df['state'].nunique()} entidades | {df['year'].min():.0f}–{df['year'].max():.0f}")

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from statsmodels.stats.diagnostic import acorr_ljungbox

Datos cargados: 10,840 filas | 52 entidades | 1999–2017


In [2]:
wv_acc = (estados[(estados['state']=='West Virginia')&
                    (estados['cause_name']=='Unintentional injuries')]
           .sort_values('year').dropna(subset=['age_adjusted_death_rate']))
serie = wv_acc['age_adjusted_death_rate'].values
anios = wv_acc['year'].values

X = np.arange(1, len(serie)+1).reshape(-1,1)
modelo_lm = LinearRegression().fit(X, serie)
y_pred = modelo_lm.predict(X)
residuos_lm = serie - y_pred

r2 = modelo_lm.score(X, serie)
rmse = np.sqrt(mean_squared_error(serie, y_pred))
mae = np.mean(np.abs(residuos_lm))
mape = np.mean(np.abs(residuos_lm/serie))*100

print(f"=== Regresión Lineal ===")
print(f"Intercepto: {modelo_lm.intercept_:.4f}")
print(f"Pendiente:  {modelo_lm.coef_[0]:.4f} por año")
print(f"R²:  {r2:.4f}")
print(f"RMSE:{rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"MAPE:{mape:.4f}%")

from statsmodels.stats.diagnostic import acorr_ljungbox
lb_lm = acorr_ljungbox(residuos_lm, lags=[4], return_df=True)
print(f"\nLjung-Box Q*: {lb_lm['lb_stat'].values[0]:.4f}, p={lb_lm['lb_pvalue'].values[0]:.4f}")
print(f"{'✔ No autocorrelación' if lb_lm['lb_pvalue'].values[0]>0.05 else '⚠ Autocorrelación significativa → supuesto OLS violado'}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(residuos_lm))), y=residuos_lm,
                          mode='lines+markers', name='Residuos',
                          line=dict(color='#6A4C93')))
fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.update_layout(title='Residuos – Regresión Lineal', height=340, template='plotly_white')
fig.show()

=== Regresión Lineal ===
Intercepto: 38.0211
Pendiente:  2.5395 por año
R²:  0.8181
RMSE:6.5581
MAE: 5.0335
MAPE:7.9567%

Ljung-Box Q*: 4.7703, p=0.3117
✔ No autocorrelación


## 13.2. Proyección – Regresión Lineal (2018–2022)

In [3]:
anios_fut = list(range(2018, 2023))
X_fut = np.arange(len(serie)+1, len(serie)+6).reshape(-1,1)
y_fut = modelo_lm.predict(X_fut)

# IC aproximado
std_err = np.sqrt(mean_squared_error(serie, y_pred))
y_fut_lo = y_fut - 2*std_err
y_fut_hi = y_fut + 2*std_err

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(anios_fut)+list(anios_fut[::-1]),
                          y=list(y_fut_hi)+list(y_fut_lo[::-1]),
                          fill='toself', fillcolor='rgba(106,76,147,0.15)',
                          line=dict(color='rgba(0,0,0,0)'), name='IC 95%'))
fig.add_trace(go.Scatter(x=list(anios), y=serie, mode='lines+markers',
                          name='Observado', line=dict(color='#3B0764', width=2)))
fig.add_trace(go.Scatter(x=list(anios), y=y_pred, mode='lines',
                          name='Ajustado', line=dict(color='#27ae60', width=1.5)))
fig.add_trace(go.Scatter(x=anios_fut, y=y_fut, mode='lines+markers',
                          name='Proyección', line=dict(color='#6A4C93', width=2, dash='dash'),
                          marker=dict(size=9, symbol='triangle-up')))
fig.add_vline(x=2017.5, line_dash='dot', line_color='gray')
fig.update_layout(title='Proyección Regresión Lineal – Unintentional Injuries · West Virginia',
                   xaxis_title='Año', yaxis_title='Tasa por 100,000 hab.',
                   height=450, template='plotly_white')
fig.show()

## 13.3. Tabla de valores proyectados

In [4]:
tabla_lm = pd.DataFrame({'Año':anios_fut,'Predicción':y_fut.round(1),
                             'IC 95% inf.':y_fut_lo.round(1),'IC 95% sup.':y_fut_hi.round(1)})
print(tabla_lm.to_string(index=False))

 Año  Predicción  IC 95% inf.  IC 95% sup.
2018        88.8         75.7        101.9
2019        91.4         78.2        104.5
2020        93.9         80.8        107.0
2021        96.4         83.3        109.5
2022        99.0         85.9        112.1
